In [1]:
!pip install -q prefect wandb ultralytics opencv-python numpy torch
!pip install "starlette>=0.49.1,<1.0.0" google-cloud-bigquery-storage
!pip install "protobuf>=3.20.3,<6.0.0"
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 651.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.9/348.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 k

In [2]:
import os
import time
import numpy as np
import cv2
import torch
import sys
from pathlib import Path
from ultralytics import YOLO
import wandb
from prefect import task, flow

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("PREFECT_API_KEY")
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")

os.environ["PREFECT_API_KEY"] = secret_value_0
os.environ["WANDB_API_KEY"] = secret_value_1

# Tự động cấu hình Prefect trỏ luồng điều phối về Cloud thay vì chạy Local
os.environ["PREFECT_API_URL"] = (
    "https://api.prefect.cloud/api/"
    "accounts/fb31a7d6-0faf-4106-902c-f7c7b19b07ad/"
    "workspaces/167bd5fa-56b2-4da2-be18-ca1959da067c"
)

start_total_time = time.time()

DATASET_ROOT = Path(
    "/kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102"
)

IMAGE_DIR = DATASET_ROOT / "unified_dataset/images"
DATA_YAML_PATH = str(DATASET_ROOT / "unified_dataset/data.yaml")

SPLIT_TXT_DIR = Path("/kaggle/working/split_metadata")

CURRENT_ACCOUNT = 3

if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dataset:\n{IMAGE_DIR}\n\n"
        "Hãy mount/copy dataset vào /kaggle/working/unified_dataset trước."
    )

print(f"✅ Dataset: {DATASET_ROOT}", flush=True)
print(f"✅ Images : {IMAGE_DIR}", flush=True)

# ============================================================
# SPLIT METADATA
#
# 3 ACCOUNT TRAIN ĐỘC LẬP
# VAL / TEST DÙNG CHUNG
# ============================================================

import random
import yaml
SPLIT_TXT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ACCOUNTS = 3
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SEED = 42

# ------------------------------------------------------------
# 1. Lấy toàn bộ ảnh
# ------------------------------------------------------------

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_images = sorted([
    p.resolve()
    for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

if not all_images:
    raise FileNotFoundError(
        f"Không tìm thấy ảnh trong: {IMAGE_DIR}"
    )

print(f"📸 Tổng số ảnh: {len(all_images):,}")

# ------------------------------------------------------------
# 2. Shuffle cố định
# ------------------------------------------------------------

rng = random.Random(SEED)
rng.shuffle(all_images)

# ------------------------------------------------------------
# 3. Chia DATASET thành 80/10/10
# ------------------------------------------------------------

total = len(all_images)

n_train = int(total * TRAIN_RATIO)
n_val = int(total * VAL_RATIO)

train_all = all_images[:n_train]
val_images = all_images[n_train:n_train + n_val]
test_images = all_images[n_train + n_val:]

print(f"TRAIN tổng : {len(train_all):,}")
print(f"VAL chung  : {len(val_images):,}")
print(f"TEST chung : {len(test_images):,}")

# ------------------------------------------------------------
# 4. Chia TRAIN đều cho 3 ACCOUNT
# ------------------------------------------------------------

account_chunks = [
    train_all[i::NUM_ACCOUNTS]
    for i in range(NUM_ACCOUNTS)
]

for account_id, account_images in enumerate(
    account_chunks,
    start=1
):
    output_file = (
        SPLIT_TXT_DIR /
        f"train_account_{account_id}.txt"
    )

    with open(output_file, "w") as f:
        for image_path in account_images:
            f.write(str(image_path) + "\n")

    print(
        f"ACC{account_id} TRAIN: "
        f"{len(account_images):,}"
    )

# ------------------------------------------------------------
# 5. VAL CHUNG
# ------------------------------------------------------------

val_file = SPLIT_TXT_DIR / "val.txt"

with open(val_file, "w") as f:
    for image_path in val_images:
        f.write(str(image_path) + "\n")

# ------------------------------------------------------------
# 6. TEST CHUNG
# ------------------------------------------------------------

test_file = SPLIT_TXT_DIR / "test.txt"

with open(test_file, "w") as f:
    for image_path in test_images:
        f.write(str(image_path) + "\n")

# ------------------------------------------------------------
# 7. CHECK KHÔNG OVERLAP
# ------------------------------------------------------------

train_sets = []

for account_id in range(1, 4):

    file = (
        SPLIT_TXT_DIR /
        f"train_account_{account_id}.txt"
    )

    with open(file, "r") as f:
        paths = {
            str(Path(line.strip()).resolve())
            for line in f
            if line.strip()
        }

    train_sets.append(paths)

for i in range(3):
    for j in range(i + 1, 3):

        overlap = train_sets[i] & train_sets[j]

        if overlap:
            raise RuntimeError(
                f"❌ ACC{i+1} và ACC{j+1} "
                f"bị trùng {len(overlap):,} ảnh!"
            )

if any(
    train_sets[i] & set(val_images)
    for i in range(3)
):
    raise RuntimeError("❌ TRAIN bị overlap với VAL!")

if any(
    train_sets[i] & set(test_images)
    for i in range(3)
):
    raise RuntimeError("❌ TRAIN bị overlap với TEST!")

print("✅ TRAIN/VAL/TEST không bị overlap.")

# ============================================================
# 8. TẠO YAML RIÊNG CHO ACCOUNT HIỆN TẠI
# ============================================================

with open(DATA_YAML_PATH, "r") as f:
    yaml_data = yaml.safe_load(f)

print("========== ORIGINAL YAML ==========")
print("nc   :", yaml_data.get("nc"))
print("names:", len(yaml_data.get("names", {})))
print("===================================")

print("========== ACCOUNT YAML ==========")
_BASE_IMG = "/kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images"
for _s in ("train", "val", "test"):
    if _s not in yaml_data:
        yaml_data[_s] = os.path.join(_BASE_IMG, _s)
print("train:", yaml_data["train"])
print("val  :", yaml_data["val"])
print("test :", yaml_data["test"])

account_train_file = (
    SPLIT_TXT_DIR /
    f"train_account_{CURRENT_ACCOUNT}.txt"
)

account_yaml = (
    SPLIT_TXT_DIR /
    f"data_account_{CURRENT_ACCOUNT}.yaml"
)

yaml_data["train"] = str(account_train_file)
yaml_data["val"] = str(val_file)
yaml_data["test"] = str(test_file)

with open(account_yaml, "w") as f:
    yaml.safe_dump(
        yaml_data,
        f,
        sort_keys=False,
        allow_unicode=True
    )

DATA_YAML_PATH = str(account_yaml)

# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

print("\n========== SPLIT METADATA ==========")

for account_id in range(1, 4):

    p = (
        SPLIT_TXT_DIR /
        f"train_account_{account_id}.txt"
    )

    with open(p, "r") as f:
        count = sum(
            1 for line in f
            if line.strip()
        )

    print(
        f"ACC{account_id} TRAIN : "
        f"{count:,}"
    )

print(f"VAL CHUNG  : {len(val_images):,}")
print(f"TEST CHUNG : {len(test_images):,}")
print(f"YAML       : {DATA_YAML_PATH}")

print("====================================\n")

# ============================================================
# KIỂM TRA SPLIT METADATA
# ============================================================
print("\n========== CHECK SPLIT METADATA ==========")

print(f"Directory: {SPLIT_TXT_DIR}")
print(f"Directory exists: {SPLIT_TXT_DIR.exists()}")

required_files = [
    SPLIT_TXT_DIR / f"train_account_{CURRENT_ACCOUNT}.txt",
    SPLIT_TXT_DIR / "val.txt",
    SPLIT_TXT_DIR / "test.txt",
]

for p in required_files:
    print(f"{p.name}: {'OK' if p.exists() else 'MISSING'}")

print("===========================================\n")

# =====================================================================
# 🔄 PREFECT TASK 1: THEO DÕI & ĐÓNG GÓI PHIÊN BẢN DATASET (W&B ARTIFACTS)
# =====================================================================
@task(name="MLOps-Data-Versioning", retries=2, retry_delay_seconds=15)
def verify_and_log_dataset_version(account_id, split_dir):
    print(f"⏳ [Prefect Task 1]: Khởi chạy xác thực tệp danh sách của máy {account_id}...", flush=True)
    
    train_txt = split_dir / f"train_account_{account_id}.txt"
    val_txt = split_dir / "val.txt"
    test_txt = split_dir / "test.txt"
    
    if not (train_txt.exists() and val_txt.exists() and test_txt.exists()):
        raise FileNotFoundError(
            f"Missing split files for account {account_id}: "
            f"{train_txt}, {val_txt}, {test_txt}"
        )
        
    # Tạo Artifacts lưu vết cố định phiên bản dữ liệu của máy lên W&B Cloud
    run = wandb.init(project="Agricultural_YOLO26s_Project", name=f"Data_Versioning_Acc_{account_id}")
    artifact = wandb.Artifact(name=f"fixed_split_account_{account_id}", type="dataset")
    artifact.add_file(str(train_txt))
    artifact.add_file(str(val_txt))
    artifact.add_file(str(test_txt))
    run.log_artifact(artifact)
    run.finish()
    
    print("✅ [Prefect Task 1]: Ghi nhận phiên bản dữ liệu lên hệ thống MLOps thành công.", flush=True)
    return test_txt

# =====================================================================
# 🚀 PREFECT TASK 2: TRAIN YOLO26s MAX CÔNG SUẤT DUAL T4 (KHÔNG OOM)
# =====================================================================
@task(name="Model-Training-Dual-T4-DDP")
def train_yolo_model(account_id, yaml_path):
    print("⏳ [Prefect Task 2]: Bật nhân GPU. Khởi chạy khối huấn luyện YOLO26s...", flush=True)
    
    # Đồng bộ luồng 'ngồi rình' tài nguyên phần cứng của W&B Web Dashboard
    wandb.init(
        project="Agricultural_YOLO26s_Project",
        name=f"Dual_T4_Account_{account_id}",
        config={
            "architecture": "YOLO26s",
            "pre_trained_weights": "/kaggle/input/datasets/vdt1501/yolo26s-objv1-150/yolo26s-objv1-150.pt",
            "batch_size": 80,
            "epochs": 50,
            "freeze_layers": 10,
            "optimizer": "MuSGD"
        }
    )

    model = YOLO("/kaggle/input/datasets/vdt1501/yolo26s-objv1-150/yolo26s-objv1-150.pt")
    
    # Thực thi tham số bẻ gãy Overfitting cho ảnh ghép 4 của bạn
    model.train(
        # --- 🎮 CẤU HÌNH PHẦN CỨNG TỐI ƯU ---
        device=[0, 1], batch=80, imgsz=640, workers=8, amp=True, cache=False, deterministic=False,
        # --- 🎯 CHỐNG OVERFIT THỰC ĐỊA ---
        data=str(yaml_path), freeze=10, patience=8, epochs=50, lr0=0.001, lrf=0.01,
        weight_decay=0.0005, warmup_epochs=2.0, label_smoothing=0.1,optimizer="MuSGD",
        # --- 🔄 LUỒNG XỬ LÝ ẢNH CHUẨN HÓA ---
        mosaic=0.3, mixup=0.05, copy_paste=0.0, translate=0.1, degrees=10.0, scale=0.15,
        fliplr=0.5, flipud=0.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, close_mosaic=10,
        project="Agricultural_YOLO26s_Project", name=f"Dual_T4_Account_{account_id}", plots=True
    )
    wandb.finish()
    print("✅ [Prefect Task 2]: Khối huấn luyện hoàn thành. Weights đã cất vào kho.", flush=True)
    return model

# =====================================================================
# 📊 PREFECT TASK 3: TÍNH TOÁN 12 CUSTOM METRIC & NÉN OPTIMUM ONNX
# =====================================================================
@task(name="Advanced-Metrics-and-Optimum-Export")
def evaluate_and_export_onnx(model, yaml_path, test_txt_path):
    print("⏳ [Prefect Task 3]: Tiến hành đánh giá Custom Metric và nén mô hình Edge...", flush=True)
    
    # Chạy kiểm thử trên tập dữ liệu Unseen Test xa lạ của riêng máy
    val_results = model.val(data=str(yaml_path), split='test', device="0,1", batch=86, plots=False)
    metrics_dict = val_results.results_dict
    
    # 1. Đọc danh sách file test để bóc tách kênh màu HSV phục vụ tính toán PSI Data Drift
    with open(test_txt_path, "r") as f:
        test_img_paths = [line.strip() for line in f.readlines() if line.strip()]
        
    brightness_values = []
    for p in test_img_paths[:150]: # Giới hạn 150 mẫu tối ưu hóa I/O CPU đọc đĩa
        if os.path.exists(p):
            v_channel = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2HSV)[:, :, 2]
            brightness_values.append(np.mean(v_channel))
    brightness_values = np.array(brightness_values) if len(brightness_values) > 0 else np.array([128.0])
    
    # Thuật toán tính chỉ số đo đạc biến động thời tiết/lệch phân phối màu sắc
    h_val, _ = np.histogram(
        brightness_values,
        bins=10,
        range=(0, 255)
    )

    h_val = h_val.astype(np.float64)
    h_val = h_val / max(h_val.sum(), 1)

    h_train = np.ones(10, dtype=np.float64) / 10

    h_val = np.clip(h_val, 1e-4, None)
    h_train = np.clip(h_train, 1e-4, None)

    psi_value = np.sum(
        (h_val - h_train) *
        np.log(h_val / h_train)
    )

        # 2. Benchmark latency GPU chính xác
    device = torch.device("cuda:0")
    model.model.to(device).half()
    model.model.eval()

    dummy_input = torch.randn(
        1, 3, 640, 640,
        device=device,
        dtype=torch.float16
    )

        # Warm-up GPU
    with torch.inference_mode():
        for _ in range(20):
            _ = model.model(dummy_input)

    torch.cuda.synchronize()

        # Benchmark
    edge_latencies = []

    with torch.inference_mode():
        for _ in range(50):
            torch.cuda.synchronize()
            t0 = time.perf_counter()

            _ = model.model(dummy_input)

            torch.cuda.synchronize()
            edge_latencies.append(
                (time.perf_counter() - t0) * 1000
            )

    p50_latency = np.percentile(edge_latencies, 50)
    p95_latency = np.percentile(edge_latencies, 95)
    p99_latency = np.percentile(edge_latencies, 99)

    # In báo cáo hệ thống khép kín
    print("\n==================================================", flush=True)
    print("📊 BÁO CÁO ĐẦU RA MLOPS PIPELINE (AUTOMATED BY PREFECT):", flush=True)
    print(f"   - mAP@0.5:0.95 (Độ chính xác nền): {metrics_dict.get('metrics/mAP50-95(B)', 0.0):.4f}", flush=True)
    print(f"   - PSI Data Drift ánh sáng thực địa: {psi_value:.4f} -> " + ("🔴 ALERT DRIFT!" if psi_value > 0.2 else "🟢 STABLE"), flush=True)
    print(f"   - Tốc độ xử lý p99 Latency trên Edge: {p99_latency:.2f} ms (Tiêu chuẩn: < 100 ms)", flush=True)
    print("==================================================", flush=True)

    # 3. Kỹ thuật Optimum: Khóa cấu hình ma trận nén mô hình sang định dạng siêu nhẹ .onnx
    print("📦 Khởi chạy Optimum nén cấu trúc sang định dạng .onnx FP16...", flush=True)
    model.export(format="onnx", half=True)
    print("✅ [Prefect Task 3]: Toàn bộ chuỗi cung ứng MLOps hoàn thành khép kín.", flush=True)

# =====================================================================
# 🎛️ KHỐI ĐIỀU PHỐI TRUNG TÂM (PREFECT FLOW ORCHESTRATION)
# =====================================================================
@flow(name="Agricultural-YOLO26s-Automation-Pipeline")
def run_agricultural_mlops_pipeline():
    # Kích hoạt luồng chạy tuyến tính được Prefect điều phối bảo vệ tài nguyên
    test_file_path = verify_and_log_dataset_version(CURRENT_ACCOUNT, SPLIT_TXT_DIR)
    trained_model = train_yolo_model(CURRENT_ACCOUNT, DATA_YAML_PATH)
    evaluate_and_export_onnx(trained_model, DATA_YAML_PATH, test_file_path)

if __name__ == "__main__":
    run_agricultural_mlops_pipeline()


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset: /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102
✅ Images : /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images
📸 Tổng số ảnh: 142,822
TRAIN tổng : 114,257
VAL chung  : 14,282
TEST chung : 14,283
ACC1 TRAIN: 38,086
ACC2 TRAIN: 38,086
ACC3 TRAIN: 38,085
✅ TRAIN/VAL/TEST không bị overlap.
========== ORIGINAL YAML ==========
nc   : 233
names: 233
========== ACCOUNT YAML ==========
train: /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images/train
val  : /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images/val
test : /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_

13:27:45.867 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8832
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

13:27:57.815 | INFO    | Flow run 'sincere-wyvern' - Beginning flow run 'sincere-wyvern' for flow 'Agricultural-YOLO26s-Automation-Pipeline'

⏳ [Prefect Task 1]: Khởi chạy xác thực tệp danh sách của máy 3...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 0220267 (vuduytien) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 10acl5qk
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260811_132758-10acl5qk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Data_Versioning_Acc_3
wandb: ⭐️ View project at https://wandb.ai/vuduytien/Agricultural_YOLO26s_Project
wandb: 🚀 View run at https://wandb.ai/vuduytien/Agricultural_YOLO26s_Project/runs/10acl5qk
wandb: updating run metadata; uploading artifact fixed_split_account_3
wandb: uploading artifact fixed_split_account_3
wandb: uploading artifact fixed_split_account_3; uploading wandb-metadata.json; uploading wandb-summary.json; uploading config.yaml
wandb: uploading artifact fixed_split_account_3
wandb: uploading data
wandb: 🚀 View run Data_Versioning_Acc_3 at: h

...

19:39:14.200 | ERROR   | Task run 'Advanced-Metrics-and-Optimum-Export-2f2' - Task run failed with exception: RuntimeError('Inference tensors do not track version counter.')
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1021, in run_context
    yield self
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1683, in run_task_sync
    engine.call_task_fn(txn)
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1038, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/3021258635.py", line 421, in evaluate_and_export_onnx
    _ = model.model(dummy_input)
        ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 157, in forward
    return self.predict(x, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 173, in predict
    return self._predict_once(x, profile, embed)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 194, in _predict_once
    x = m(x)  # run
        ^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py", line 89, in forward_fuse
    return self.act(self.conv(x))
                    ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py", line 553, in forward
    return self._conv_forward(input, self.weight, self.bias)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py", line 548, in _conv_forward
    return F.conv2d(
           ^^^^^^^^^
RuntimeError: Inference tensors do not track version counter.

19:39:14.214 | ERROR   | Task run 'Advanced-Metrics-and-Optimum-Export-2f2' - Finished in state Failed('Task run encountered an exception RuntimeError: Inference tensors do not track version counter.')

19:39:14.855 | ERROR   | Flow run 'sincere-wyvern' - Encountered exception during execution: RuntimeError('Inference tensors do not track version counter.')
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/prefect/flow_engine.py", line 1277, in run_context
    yield self
  File "/usr/local/lib/python3.12/dist-packages/prefect/flow_engine.py", line 2030, in run_flow_sync
    engine.call_flow_fn()
  File "/usr/local/lib/python3.12/dist-packages/prefect/flow_engine.py", line 1297, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/3021258635.py", line 465, in run_agricultural_mlops_pipeline
    evaluate_and_export_onnx(trained_model, DATA_YAML_PATH, test_file_path)
  File "/usr/local/lib/python3.12/dist-packages/prefect/tasks.py", line 1227, in __call__
    return run_task(
           ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1916, in run_task
    return run_task_sync(**kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1686, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 617, in result
    raise self._raised
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1021, in run_context
    yield self
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1683, in run_task_sync
    engine.call_task_fn(txn)
  File "/usr/local/lib/python3.12/dist-packages/prefect/task_engine.py", line 1038, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/3021258635.py", line 421, in evaluate_and_export_onnx
    _ = model.model(dummy_input)
        ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 157, in forward
    return self.predict(x, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 173, in predict
    return self._predict_once(x, profile, embed)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 194, in _predict_once
    x = m(x)  # run
        ^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py", line 89, in forward_fuse
    return self.act(self.conv(x))
                    ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, 

19:39:14.886 | INFO    | Flow run 'sincere-wyvern' - Finished in state Failed('Flow run encountered an exception: RuntimeError: Inference tensors do not track version counter.')

RuntimeError: Inference tensors do not track version counter.